# routes/browser

> Route handlers for file browser navigation and file selection

In [ ]:
#| default_exp routes.browser

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Any, List, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile
from cjm_transcription_source_select.utils import detect_file_type, is_media_file
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection, render_browser_panel
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.stats_panel import render_stats_panel

## Navigate Handler

Updates the current directory and re-renders the file browser.

In [ ]:
#| export
def _handle_navigate(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # Directory path to navigate to
):  # Rendered browser panel
    """Navigate to a directory and re-render the browser panel."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    browser_state = get_browser_state(step_state, home_path)

    # Validate path
    valid, _ = provider.is_valid_path(path)
    if valid and provider.is_directory(path):
        browser_state.current_path = provider.normalize_path(path)
    else:
        browser_state.current_path = step_state.get("current_directory", home_path)

    # Sync selection with selected_files and selected_folders
    selected_files = step_state.get("selected_files", [])
    selected_folders = step_state.get("selected_folders", [])
    sync_browser_selection(browser_state, selected_files, selected_folders)

    # Save state
    update_step_state(
        state_store, workflow_id, session_id,
        current_directory=browser_state.current_path,
        browser_state=browser_state.to_dict(),
    )

    return render_browser_panel(
        browser_state=browser_state,
        config=config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )

## Folder Media Files Helper

Lists media files (shallow) in a directory for folder bulk-select.

In [ ]:
#| export
def _list_media_in_folder(
    folder_path: str,  # Directory to scan for media files
) -> List[SelectedFile]:  # Media files found as SelectedFile dicts
    """List media files in a directory (shallow, no recursion)."""
    folder = Path(folder_path)
    if not folder.is_dir():
        return []
    results = []
    for child in sorted(folder.iterdir()):
        if child.is_file() and is_media_file(str(child)):
            file_type = detect_file_type(str(child))
            if file_type:
                results.append(SelectedFile(
                    path=str(child),
                    filename=child.name,
                    file_type=file_type,
                    size_bytes=child.stat().st_size,
                    duration=None,
                    format=child.suffix.lstrip("."),
                ))
    return results

## Select Handler

Toggles a file or folder in/out of the selection. For files, toggles the individual file. For folders, bulk-adds or bulk-removes all media files in that directory (shallow). Returns the browser panel as the primary swap target, plus OOB swaps for the selection and stats panels.

In [ ]:
#| export
def _handle_select(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File or folder path to toggle
):  # (browser panel, OOB selection panel, OOB stats panel)
    """Toggle a file or folder in/out of the selected files list."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    browser_state = get_browser_state(step_state, home_path)
    selected_files = step_state.get("selected_files", [])
    selected_folders = step_state.get("selected_folders", [])
    extraction_results = step_state.get("extraction_results", {})

    target = Path(path)

    if target.is_dir():
        # Folder toggle: bulk add/remove all media files in directory
        if path in selected_folders:
            # Remove all files from this folder and deselect it
            folder_file_paths = {str(child) for child in target.iterdir()
                                 if child.is_file() and is_media_file(str(child))}
            selected_files = [f for f in selected_files if f["path"] not in folder_file_paths]
            for fp in folder_file_paths:
                extraction_results.pop(fp, None)
            selected_folders = [f for f in selected_folders if f != path]
        else:
            # Add media files not already selected; skip folders with no media files
            all_media = _list_media_in_folder(path)
            if all_media:
                existing_paths = {f["path"] for f in selected_files}
                new_files = [f for f in all_media if f["path"] not in existing_paths]
                selected_files.extend(new_files)
                selected_folders.append(path)
    else:
        # Single file toggle (existing behavior)
        existing_paths = {f["path"] for f in selected_files}
        if path in existing_paths:
            selected_files = [f for f in selected_files if f["path"] != path]
            extraction_results.pop(path, None)
            # Auto-deselect parent folder if no files from it remain
            parent = str(target.parent)
            if parent in selected_folders:
                remaining = {f["path"] for f in selected_files}
                has_sibling = any(str(Path(p).parent) == parent for p in remaining)
                if not has_sibling:
                    selected_folders = [f for f in selected_folders if f != parent]
        else:
            file_type = detect_file_type(path)
            if file_type and target.exists():
                selected_files.append(SelectedFile(
                    path=path,
                    filename=target.name,
                    file_type=file_type,
                    size_bytes=target.stat().st_size,
                    duration=None,
                    format=target.suffix.lstrip("."),
                ))

    # Sync browser selection with updated lists
    sync_browser_selection(browser_state, selected_files, selected_folders)

    # Clear verified state when selection changes
    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        selected_folders=selected_folders,
        extraction_results=extraction_results,
        verified=False,
        committed_audio_paths=[],
        browser_state=browser_state.to_dict(),
    )

    # Primary swap: browser panel
    browser = render_browser_panel(
        browser_state=browser_state,
        config=config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )

    # OOB swap: selection panel
    selection = render_selection_panel(selected_files, urls, extraction_results)
    selection.attrs["hx-swap-oob"] = "outerHTML"

    # OOB swap: stats panel
    stats = render_stats_panel(selected_files, urls, extraction_results, verified=False)
    stats.attrs["hx-swap-oob"] = "outerHTML"

    return browser, selection, stats

## Router Initialization

In [ ]:
#| export
def init_browser_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle (populated after router init)
    home_path: str = "",  # Home directory for nav buttons
    prefix: str = "/browser",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize the file browser router with navigate and select handlers."""
    router = APIRouter(prefix=prefix)
    _home = home_path or provider.get_home_path()

    @router
    def navigate(sess, path: str = ""):
        """Navigate to a directory."""
        target = path or _home
        return _handle_navigate(
            state_store, provider, config, workflow_id, _home, urls,
            sess, target,
        )

    @router
    def select(sess, path: str = ""):
        """Toggle a file in/out of the selection."""
        if not path:
            return navigate(sess)
        return _handle_select(
            state_store, provider, config, workflow_id, _home, urls,
            sess, path,
        )

    routes = {
        "navigate": navigate,
        "select": select,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()